# CS570 — Project Deliverable 1: Data Loading & Exploratory Analysis
**Team:** [team name]  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE ,MIR AHMAD ALI , MIR AHMAD ALI , RAMESH MANDAMANEDI , YUEXUAN LU  
**Date:** February 25, 2026


In [1]:
import os

# ── Point to the ml-1m folder in the project directory ──────────
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'ml-1m')
# ───────────────────────────────────────────────────────────────

RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.dat')
USERS_PATH   = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH  = os.path.join(DATA_DIR, 'movies.dat')

for path in [RATINGS_PATH, USERS_PATH, MOVIES_PATH]:
    status = 'found' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status}: {path}')


found: /Users/azatbekismailov/Desktop/SFBU/Big data /Project/ml-1m/ratings.dat
found: /Users/azatbekismailov/Desktop/SFBU/Big data /Project/ml-1m/users.dat
found: /Users/azatbekismailov/Desktop/SFBU/Big data /Project/ml-1m/movies.dat


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D1-MovieLens')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/25 20:10:11 WARN Utils: Your hostname, Azatbeks-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.16.9.176 instead (on interface en0)
26/02/25 20:10:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 20:10:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/25 20:10:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1


## 1. Data Loading
Each file is loaded with an **explicit schema** using `StructType`/`StructField`. No `inferSchema=True`.


In [3]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])

USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])

MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])
print('Schemas defined.')


Schemas defined.


In [4]:
# ratings.dat
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_PATH)
ratings.printSchema()
print('Row count:', ratings.count())
ratings.show(5)


root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)

Row count: 1000209
+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|   5.0|978300760|
|     1|    661|   3.0|978302109|
|     1|    914|   3.0|978301968|
|     1|   3408|   4.0|978300275|
|     1|   2355|   5.0|978824291|
+------+-------+------+---------+
only showing top 5 rows


In [5]:
# users.dat
users = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
users.printSchema()
print('Row count:', users.count())
users.show(5)


root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)

Row count: 6040
+------+------+---+----------+-------+
|UserID|Gender|Age|Occupation|ZipCode|
+------+------+---+----------+-------+
|     1|     F|  1|        10|  48067|
|     2|     M| 56|        16|  70072|
|     3|     M| 25|        15|  55117|
|     4|     M| 45|         7|  02460|
|     5|     M| 25|        20|  55455|
+------+------+---+----------+-------+
only showing top 5 rows


In [6]:
# movies.dat
movies = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)
movies.printSchema()
print('Row count:', movies.count())
movies.show(5)


root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

Row count: 3883
+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


## 2. Join the Tables
`ratings` ↔ `users` on **UserID** · `ratings` ↔ `movies` on **MovieID** · both `inner` joins.


In [7]:
joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
)

print('Row count:   ', joined.count())
print('Column count:', len(joined.columns))
joined.printSchema()
joined.show(5)


Row count:    1000209
Column count: 10
root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|   1193|     1|   5.0|978300760|     F|  1|        10|  48067|One Flew Over the...|               Drama|
|    661|     1|   3.0|978302109|     F|  1|        10|  48067|James and the Gia...|Animation|Childre...|
|    914|     1|   3.0|978301968|     F

## 3. Basic Statistics


In [8]:
joined.describe().show()


+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|summary|           MovieID|            UserID|            Rating|           Timestamp| Gender|               Age|       Occupation|           ZipCode|               Title| Genres|
+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|  count|           1000209|           1000209|           1000209|             1000209|1000209|           1000209|          1000209|           1000209|             1000209|1000209|
|   mean|1865.5398981612843| 3024.512347919285| 3.581564453029317| 9.722436954046655E8|   NULL| 29.73831369243828|8.036138447064564| 223239.8917114074|                NULL|   NULL|
| stddev|1096.0406894572482|1728.4126948999715|1.1171018453732606|1.2152558939916052E7|   NULL|

### Observations

The rating range is **1.0 to 5.0** (integer-only, no half-stars), with a mean of **3.58** and a standard deviation of **1.12**. 
The mean being well above the neutral midpoint of 3.0 reveals a **positive rating bias** — users are more likely to rate movies they enjoyed, which is a classic self-selection effect in recommender system datasets. 
The `Timestamp` column stands out as unusual: its mean (≈ 9.72 × 10⁸) and standard deviation (≈ 1.22 × 10⁷) are raw Unix epoch values that appear as large, unreadable numbers in the `describe()` output — these need to be converted to human-readable dates (the data spans April 2000 to February 2003) before any meaningful temporal analysis can be done.


#### EDA 1 — Rating Distribution


In [ ]:
# Rating distribution — are users generous or critical?
from pyspark.sql import functions as F
from pyspark.sql.window import Window

total = joined.count()
rating_dist = (
    joined
    .groupBy('Rating')
    .agg(F.count('*').alias('Count'))
    .withColumn('Percentage', F.round(F.col('Count') / total * 100, 1))
    .withColumn('Bar', F.repeat(F.lit('█'), (F.col('Count') / total * 50).cast('int')))
    .orderBy('Rating')
)
rating_dist.show(truncate=False)


#### EDA 2 — Top 10 Highest-Rated Movies (≥ 100 ratings)
Filtering by minimum 100 ratings avoids obscure films with a handful of perfect scores.


In [ ]:
# Top 10 highest-rated movies with statistical significance
top_rated = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.count('*').alias('Num_Ratings'),
    )
    .filter(F.col('Num_Ratings') >= 100)
    .orderBy(F.desc('Avg_Rating'))
)
top_rated.show(10, truncate=False)


#### EDA 3 — Gender Rating Patterns
Do male and female users rate differently?


In [ ]:
# Average rating by gender + volume
gender_stats = (
    joined
    .groupBy('Gender')
    .agg(
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 3).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 3).alias('Std_Rating'),
        F.countDistinct('UserID').alias('Unique_Users'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Unique_Users'), 1))
    .orderBy('Gender')
)
gender_stats.show(truncate=False)


#### EDA 4 — Age Group Rating Behavior
Age codes: 1=Under 18, 18=18-24, 25=25-34, 35=35-44, 45=45-49, 50=50-55, 56=56+


In [ ]:
# Rating behavior by age group
age_labels = {1:'Under 18', 18:'18-24', 25:'25-34', 35:'35-44', 45:'45-49', 50:'50-55', 56:'56+'}
from pyspark.sql.functions import create_map, lit
mapping = create_map([val for k, v in age_labels.items() for val in (lit(k), lit(v))])

age_stats = (
    joined
    .withColumn('Age_Group', mapping[F.col('Age')])
    .groupBy('Age', 'Age_Group')
    .agg(
        F.countDistinct('UserID').alias('Users'),
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Users'), 1))
    .orderBy('Age')
)
age_stats.show(truncate=False)


#### EDA 5 — Genre Popularity vs Quality
Which genres are most watched vs. most loved?


In [ ]:
# Genre analysis: popularity (count) vs quality (avg rating)
genre_stats = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('Genre'), 'Rating')
    .groupBy('Genre')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 2).alias('Std_Rating'),
    )
    .orderBy(F.desc('Num_Ratings'))
)
genre_stats.show(20, truncate=False)


#### EDA 6 — User Activity Distribution
Is there a power-law pattern in user engagement?


In [ ]:
# User activity — classify into engagement tiers
user_activity = ratings.groupBy('UserID').agg(F.count('*').alias('num_ratings'))

user_tiers = (
    user_activity
    .withColumn('Tier', F.when(F.col('num_ratings') < 50, 'Light (< 50)')
                         .when(F.col('num_ratings') < 150, 'Medium (50-149)')
                         .when(F.col('num_ratings') < 500, 'Active (150-499)')
                         .otherwise('Power (500+)'))
    .groupBy('Tier')
    .agg(
        F.count('*').alias('Users'),
        F.sum('num_ratings').alias('Total_Ratings'),
        F.round(F.avg('num_ratings'), 1).alias('Avg_Ratings_Per_User'),
    )
    .orderBy('Avg_Ratings_Per_User')
)
user_tiers.show(truncate=False)

# Quick stats
print('User activity summary:')
user_activity.select(
    F.min('num_ratings').alias('Min'),
    F.expr('percentile_approx(num_ratings, 0.25)').alias('Q1'),
    F.expr('percentile_approx(num_ratings, 0.5)').alias('Median'),
    F.expr('percentile_approx(num_ratings, 0.75)').alias('Q3'),
    F.max('num_ratings').alias('Max'),
    F.round(F.avg('num_ratings'), 1).alias('Mean'),
).show(truncate=False)


#### EDA 7 — Rating Trends Over Time
How does rating volume and average change over the data collection period?


In [ ]:
# Monthly rating trends
temporal = (
    joined
    .withColumn('date', F.from_unixtime('Timestamp'))
    .withColumn('YearMonth', F.date_format('date', 'yyyy-MM'))
    .groupBy('YearMonth')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.countDistinct('UserID').alias('Active_Users'),
    )
    .orderBy('YearMonth')
)
temporal.show(50, truncate=False)


## 4. EDA Questions


In [9]:
# A. Unique genres (explode pipe-separated values)
from pyspark.sql import functions as F

unique_genres = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('genre'))
    .distinct()
    .count()
)
print(f'A. Unique individual genres: {unique_genres}')


A. Unique individual genres: 18


In [10]:
# B. Average rating — age group 25-34 (Age code = 25)
avg_25_34 = (
    joined
    .filter(F.col('Age') == 25)
    .agg(F.round(F.avg('Rating'), 2).alias('avg_rating'))
    .collect()[0]['avg_rating']
)
print(f'B. Average rating (25-34 age group): {avg_25_34}')


B. Average rating (25-34 age group): 3.55


In [11]:
# C. Movie with the most ratings
top = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(F.count('*').alias('rating_count'))
    .orderBy(F.desc('rating_count'))
    .first()
)
print(f'C. Most rated movie : {top["Title"]}')
print(f'   Number of ratings: {top["rating_count"]}')


C. Most rated movie : American Beauty (1999)
   Number of ratings: 3428


## 5. Data Quality Observations


In [12]:
# Issue 1: Raw Timestamp — hard to read
joined.agg(F.min('Timestamp'), F.max('Timestamp')).show()
joined.select(F.from_unixtime('Timestamp').alias('readable_date')).show(5)


+--------------+--------------+
|min(Timestamp)|max(Timestamp)|
+--------------+--------------+
|     956703932|    1046454590|
+--------------+--------------+

+-------------------+
|      readable_date|
+-------------------+
|2000-12-31 14:12:40|
|2000-12-31 14:35:09|
|2000-12-31 14:32:48|
|2000-12-31 14:04:35|
|2001-01-06 15:38:11|
+-------------------+
only showing top 5 rows


In [ ]:
# Issue 2: Non-standard Zip Codes (6-digit and 9-digit codes)
total_users = users.count()

# Find zip codes that are NOT exactly 5 digits
non_standard = users.filter(~F.col('ZipCode').rlike(r'^\d{5}$'))
non_standard_count = non_standard.count()

# Classify by length
non_standard_with_len = non_standard.withColumn('zip_length', F.length('ZipCode'))

print(f'Total users:                {total_users:,}')
print(f'Non-standard zip codes:     {non_standard_count}')
print()

# Show breakdown by zip code length
print('Breakdown by zip code length:')
non_standard_with_len.groupBy('zip_length').count().orderBy('zip_length').show()

# Specifically highlight 6-digit and 9-digit codes
print('6-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 6).select('UserID', 'ZipCode').show(truncate=False)

print('9-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 9).select('UserID', 'ZipCode').show(truncate=False)


In [ ]:
# Issue 3: Movies with zero ratings (orphan movies)
# Count ratings per movie
rating_counts = ratings.groupBy('MovieID').agg(F.count('*').alias('Ratings'))

# Left join movies with rating counts — orphans will have null Ratings
movies_with_counts = movies.join(rating_counts, on='MovieID', how='left').fillna(0, subset=['Ratings'])

# Filter orphan movies (zero ratings)
orphan_movies = movies_with_counts.filter(F.col('Ratings') == 0)
orphan_count = orphan_movies.count()
total_movies = movies.count()

print(f'Total movies in dataset:       {total_movies:,}')
print(f'Movies with zero ratings:      {orphan_count}')
print(f'Movies with at least 1 rating: {total_movies - orphan_count:,}')
print()
print('Sample orphan movies (no ratings):')
orphan_movies.select('MovieID', 'Title', 'Genres', 'Ratings').orderBy('MovieID').show(10, truncate=False)


### Issues Found

**Issue 1 — Raw Unix timestamps are not human-readable.**  
The `Timestamp` column stores ratings as raw Unix epoch integers (e.g., `978300760`), which are not interpretable at a glance. The timestamps range from **956,703,932** (April 25, 2000) to **1,046,454,590** (February 28, 2003). While the values are valid and contain no negatives or zeros, they need to be converted to proper datetime format for any time-based analysis such as trend detection or seasonal patterns.  
- Handle in D2: Convert the `Timestamp` column to a readable datetime using `F.from_unixtime('Timestamp')` and extract useful features such as year, month, day of week, and hour for temporal analysis.

**Issue 2 — Non-standard zip code formats (6-digit and 9-digit codes).**  
Standard US zip codes are exactly 5 digits (e.g., `48067`). Our analysis found entries with **6 digits** (e.g., `111225`) and **9 digits** (e.g., `193122042`) that do not correspond to any valid US postal format. These are likely **data entry errors** — for example, a user may have accidentally typed an extra digit, or concatenated a ZIP+4 code without the dash. This is a problem because:  
1. These zip codes **cannot be mapped to real geographic locations**, making location-based analysis unreliable.  
2. They will **fail to join** with any external geographic lookup table (e.g., zip-to-state mapping), causing data loss.  
3. If used in grouping or aggregation, they will create **incorrect or orphan categories** that skew results.  
- Handle in D2: Truncate all zip codes to the first 5 characters using `F.substring('ZipCode', 1, 5)`, or flag the malformed entries and exclude them from geographic analysis.

**Issue 3 — 177 movies in the catalog have zero ratings.**  
By left-joining the movies table with a per-movie rating count, we found that **177 movies** have a `Ratings` count of **0** — they exist in the catalog but have never been rated by any user. These orphan records are problematic because:  
1. They are **unusable for collaborative filtering**, since the algorithm requires at least some user–item interactions to generate recommendations.  
2. They **inflate the item space** unnecessarily, increasing computation without adding predictive value.  
3. They could introduce **cold-start bias** if included in evaluation metrics, making model performance appear worse than it is.  
- Handle in D2: Filter out movies with zero ratings before model training, or flag them separately for a cold-start handling strategy.


## 6. Contribution Statement

**AZATBEK ISMAILOV:** I contributed to the data quality observations section (Section 5). I identified three key issues in the dataset: (1) raw Unix timestamps that are not human-readable and need conversion for temporal analysis, (2) non-standard zip code formats including 6-digit and 9-digit codes that are likely data entry errors and would break geographic lookups, and (3) 177 orphan movies in the catalog with zero ratings that are unusable for collaborative filtering. For each issue, I wrote the PySpark queries to detect and quantify the problem, and proposed handling strategies for Deliverable 2.

**FSEHAYE MEDHANIE:** I contributed...

**MIR AHMAD ALI:** I contributed...

**RAMESH MANDAMANEDI:** I contributed...

**YUEXUAN LU:** I contributed for completing Part 4 (Exploratory Data Analysis), including data cleaning, visualization, and answering all EDA questions. Through my partner’s work, I gained a better understanding of how modeling techniques build upon exploratory analysis and how different components of a data project connect together. This collaboration helped me see the full data analysis pipeline more clearly.